# Wellbore Pseudo-Top Regime Mean Submission

## 1. Setup

In [ ]:
"Imports, constants, and robust data discovery."

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")
LOCAL_ROOT = Path.cwd()

MODEL_NAME = "pseudo_top_regime_mean_confidence_s100"
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
SEED = 20260630
MAX_TARGET_ROWS_PER_WELL = 180
ENSEMBLE_SEEDS = [20260630, 20260631]

def find_competition_base():
    if KAGGLE_INPUT_ROOT.exists():
        for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv"):
            base = path.parent
            if (base / "train").exists() and (base / "test").exists():
                return base
    for root in [LOCAL_ROOT, *LOCAL_ROOT.parents]:
        base = root / "data" / "raw"
        if (base / "sample_submission.csv").exists() and (base / "train").exists() and (base / "test").exists():
            return base
    raise FileNotFoundError("Could not locate competition files.")

BASE_DIR = find_competition_base()
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
SAMPLE_PATH = BASE_DIR / "sample_submission.csv"
OUTPUT_PATH = KAGGLE_OUTPUT / "submission.csv" if KAGGLE_OUTPUT.exists() else Path("submission.csv")

print("base_dir:", BASE_DIR)
print("model:", MODEL_NAME)

## 2. Feature Helpers

In [ ]:
"Data loading, descriptors, and row-level feature construction."

def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def load_horizontal(path):
    df = pd.read_csv(path).sort_values("MD").reset_index(drop=True)
    df.index.name = "row_idx"
    return df


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def load_sample_submission(path):
    sample = pd.read_csv(path)
    split = sample["id"].str.rsplit("_", n=1, expand=True)
    sample["wellname"] = split[0]
    sample["row_idx"] = split[1].astype(int)
    return sample


def safe_corr(left, right):
    a = np.asarray(left, dtype=float)
    b = np.asarray(right, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if int(mask.sum()) < 5:
        return np.nan
    if float(np.nanstd(a[mask])) < 1e-9 or float(np.nanstd(b[mask])) < 1e-9:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def linear_slope(x_values, y_values):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    return float(np.polyfit(x[mask], y[mask], 1)[0])


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_descriptor(wellname, directory):
    horizontal = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv")
    typewell = load_typewell(directory, wellname)
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    if known.empty:
        known = horizontal.head(1).copy()
    known_tail = known.tail(300).copy()
    gr_tail = known_tail["GR"].astype(float).clip(0, 300)
    last_known_tvt = float(known["TVT_input"].dropna().iloc[-1]) if known["TVT_input"].notna().any() else float(known["TVT"].dropna().iloc[-1])
    known_tvt = known["TVT_input"].ffill().fillna(known.get("TVT", pd.Series(index=known.index, dtype=float))).astype(float)
    typewell_gr = interp_typewell_gr(typewell, known_tvt.tail(len(gr_tail)).to_numpy())

    lag_grid = np.arange(-40.0, 42.5, 2.5)
    lag_rows = []
    for lag in lag_grid:
        lag_gr = interp_typewell_gr(typewell, known_tvt.tail(len(gr_tail)).to_numpy() + lag)
        lag_rows.append((lag, safe_corr(gr_tail.to_numpy(), lag_gr), np.nanmean(np.abs(gr_tail.to_numpy() - lag_gr))))
    lag_frame = pd.DataFrame(lag_rows, columns=["lag", "corr", "mae"])
    best_lag_row = lag_frame.sort_values(["corr", "mae"], ascending=[False, True]).iloc[0] if len(lag_frame) else pd.Series({"lag": 0.0, "corr": np.nan, "mae": np.nan})

    md_span = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    tvt_span = float(known_tvt.iloc[-1] - known_tvt.iloc[0]) if len(known_tvt) > 1 else 0.0
    xy_span = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    z_drop = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    eval_rows = int(horizontal["TVT_input"].isna().sum())
    eval_md_span = float(horizontal["MD"].iloc[-1] - known["MD"].iloc[-1]) if len(horizontal) else 0.0
    extrapolation_stress = float(np.log1p(max(eval_md_span, 0.0)) + np.log1p(max(eval_rows, 0)))

    return {
        "wellname": wellname,
        "known_rows": int(len(known)),
        "known_tvt_span": tvt_span,
        "known_md_span": md_span,
        "known_gr_mean_last300": float(gr_tail.mean()) if gr_tail.notna().any() else np.nan,
        "known_gr_std_last300": float(gr_tail.std()) if gr_tail.notna().sum() > 2 else 0.0,
        "known_gr_slope_last300": linear_slope(np.arange(len(gr_tail)), gr_tail.to_numpy()),
        "xy_span_known": xy_span,
        "z_drop_known": z_drop,
        "typewell_corr_last300": safe_corr(gr_tail.to_numpy(), typewell_gr),
        "typewell_mae_last300": float(np.nanmean(np.abs(gr_tail.to_numpy() - typewell_gr))) if np.isfinite(typewell_gr).any() else np.nan,
        "typewell_best_lag_corr": float(best_lag_row["corr"]),
        "typewell_best_lag_tvt": float(best_lag_row["lag"]),
        "typewell_best_lag_mae": float(best_lag_row["mae"]),
        "extrapolation_stress_score": extrapolation_stress,
    }


DESCRIPTOR_BASE_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "extrapolation_stress_score",
]


def add_analog_descriptors(train_desc, test_desc):
    train_desc = train_desc.copy()
    test_desc = test_desc.copy()
    all_desc = pd.concat([train_desc.assign(_split="train"), test_desc.assign(_split="test")], ignore_index=True)
    X_all = all_desc[DESCRIPTOR_BASE_COLS].replace([np.inf, -np.inf], np.nan)
    X_all = SimpleImputer(strategy="median").fit_transform(X_all)
    X_all = StandardScaler().fit_transform(X_all)
    train_idx = all_desc["_split"].eq("train").to_numpy()
    dist_all = pairwise_distances(X_all, X_all)
    train_positions = np.flatnonzero(train_idx)
    for pos, global_idx in enumerate(train_positions):
        dist_all[global_idx, global_idx] = np.inf
    nearest_train_dist = dist_all[train_idx][:, train_idx].min(axis=1)
    nearest_test_dist = dist_all[~train_idx][:, train_idx].min(axis=1) if (~train_idx).any() else np.array([])
    train_desc["selected_analog_distance"] = nearest_train_dist
    test_desc["selected_analog_distance"] = nearest_test_dist
    for frame in [train_desc, test_desc]:
        frame["analog_stress_score"] = np.log1p(np.clip(frame["selected_analog_distance"].astype(float), 0, None))
        frame["combined_stress_score"] = frame["extrapolation_stress_score"].astype(float) + frame["analog_stress_score"].astype(float)
        frame["descriptor_cluster"] = pd.qcut(frame["combined_stress_score"].rank(method="first"), q=5, labels=False, duplicates="drop").astype(float)
    return train_desc, test_desc


DESCRIPTOR_COLS = [*DESCRIPTOR_BASE_COLS, "selected_analog_distance", "analog_stress_score", "combined_stress_score", "descriptor_cluster"]


def safe_divide(num, den):
    return np.asarray(num, dtype=float) / (float(den) + 1e-9)


def build_feature_rows(wellname, directory, descriptor_row, row_indices=None, require_target=True, max_target_rows=None):
    df = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv")
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    if row_indices is None:
        target = df.loc[last_known_idx + 1:].copy()
        if require_target:
            target = target.loc[target["TVT"].notna()].copy()
        if max_target_rows is not None and len(target) > max_target_rows:
            sample_idx = np.linspace(0, len(target) - 1, max_target_rows).round().astype(int)
            target = target.iloc[np.unique(sample_idx)].copy()
    else:
        target = df.loc[list(row_indices)].copy()
    if target.empty:
        return pd.DataFrame()

    typewell = load_typewell(directory, wellname)
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    finite_known_gr = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    known_md_span = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_series = known["TVT_input"].ffill().fillna(known.get("TVT", pd.Series(index=known.index, dtype=float))).astype(float)
    known_tvt_delta = float(known_tvt_series.iloc[-1] - known_tvt_series.iloc[0]) if len(known_tvt_series) > 1 else 0.0

    out = pd.DataFrame({
        "wellname": wellname,
        "row_idx_original": target.index.astype(float),
        "MD": target["MD"].astype(float).to_numpy(),
        "X": target["X"].astype(float).to_numpy(),
        "Y": target["Y"].astype(float).to_numpy(),
        "Z": target["Z"].astype(float).to_numpy(),
        "GR": target_gr.to_numpy(),
        "last_known_tvt_feature": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last.get("TVT", 0.0)),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(known_tvt_series.iloc[-1]),
        "state_known_tvt_slope": known_tvt_delta / max(abs(known_md_span), 1.0),
        "state_known_gr_coverage": float(known_gr.notna().mean()) if len(known_gr) else 0.0,
        "state_known_gr_last": float(finite_known_gr.iloc[-1]) if len(finite_known_gr) else gr_fill,
        "state_known_gr_iqr": float(finite_known_gr.quantile(0.75) - finite_known_gr.quantile(0.25)) if len(finite_known_gr) >= 4 else 0.0,
        "state_known_xy_span": float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0,
        "state_known_z_drop": float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0,
        "state_known_md_span": known_md_span,
    })
    if "TVT" in target.columns and target["TVT"].notna().any():
        out["TVT"] = target["TVT"].astype(float).to_numpy()
        out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(descriptor_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(descriptor_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]
    for col in DESCRIPTOR_COLS:
        out[col] = float(descriptor_row[col]) if col in descriptor_row and pd.notna(descriptor_row[col]) else np.nan
    if require_target:
        for formation in FORMATION_COLS:
            out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
    return out

## 3. Model Fit

In [ ]:
"Fit pseudo-top models, global XGBoost mean, and regime-specific Ridge means."

train_wells = discover_wells(TRAIN_DIR)
test_wells = discover_wells(TEST_DIR)
train_desc = pd.DataFrame([build_descriptor(well, TRAIN_DIR) for well in train_wells])
test_desc = pd.DataFrame([build_descriptor(well, TEST_DIR) for well in test_wells])
train_desc, test_desc = add_analog_descriptors(train_desc, test_desc)

train_rows = []
for row in train_desc.to_dict(orient="records"):
    frame = build_feature_rows(row["wellname"], TRAIN_DIR, row, require_target=True, max_target_rows=MAX_TARGET_ROWS_PER_WELL)
    if not frame.empty:
        train_rows.append(frame)
train_df = pd.concat(train_rows, ignore_index=True)

sample = load_sample_submission(SAMPLE_PATH)
test_desc_map = {row["wellname"]: row for row in test_desc.to_dict(orient="records")}
test_rows = []
for wellname, group in sample.groupby("wellname", sort=True):
    descriptor = test_desc_map.get(wellname)
    if descriptor is None:
        descriptor = build_descriptor(wellname, TEST_DIR)
    frame = build_feature_rows(wellname, TEST_DIR, descriptor, row_indices=group["row_idx"].tolist(), require_target=False)
    if not frame.empty:
        test_rows.append(frame)
test_df = pd.concat(test_rows, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
TYPEWELL_FEATURE_COLS = ["tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag"]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
BASE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS, *TYPEWELL_FEATURE_COLS] if col in train_df.columns]
REGIME_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS, *TYPEWELL_FEATURE_COLS, *STATE_FEATURE_COLS] if col in train_df.columns]
PSEUDO_TOP_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS, *TYPEWELL_FEATURE_COLS, *STATE_FEATURE_COLS] if col in train_df.columns]

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 240,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": -1,
    "verbosity": 0,
}

X_top_train = train_df[PSEUDO_TOP_COLS].replace([np.inf, -np.inf], np.nan)
X_top_test = test_df[PSEUDO_TOP_COLS].replace([np.inf, -np.inf], np.nan)
for formation in FORMATION_COLS:
    target_col = f"formation_top_{formation}"
    label_mask = train_df[target_col].notna()
    median_top = float(train_df.loc[label_mask, target_col].median()) if label_mask.any() else float(train_df["Z"].median())
    seed_predictions = []
    if int(label_mask.sum()) >= 20:
        for seed in ENSEMBLE_SEEDS:
            model = XGBRegressor(**pseudo_params, random_state=seed)
            model.fit(X_top_train.loc[label_mask], train_df.loc[label_mask, target_col].astype(float), verbose=False)
            seed_predictions.append(model.predict(X_top_test))
        pred_matrix = np.column_stack(seed_predictions)
        test_df[f"pseudo_top_elev_{formation}"] = pred_matrix.mean(axis=1)
        test_df[f"pseudo_top_std_{formation}"] = pred_matrix.std(axis=1)
    else:
        test_df[f"pseudo_top_elev_{formation}"] = median_top
        test_df[f"pseudo_top_std_{formation}"] = 0.0
    train_df[f"pseudo_top_elev_{formation}"] = train_df[target_col].astype(float).fillna(median_top)
    train_df[f"pseudo_top_std_{formation}"] = 0.0
    for frame in [train_df, test_df]:
        frame[f"pseudo_signed_offset_{formation}"] = frame["Z"] - frame[f"pseudo_top_elev_{formation}"]
        frame[f"pseudo_abs_offset_{formation}"] = np.abs(frame[f"pseudo_signed_offset_{formation}"])


def add_regime_probabilities(frame, scale=50.0):
    abs_cols = [f"pseudo_abs_offset_{formation}" for formation in FORMATION_COLS]
    abs_matrix = frame[abs_cols].to_numpy(float)
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    probs = np.exp(logits)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    for idx, formation in enumerate(FORMATION_COLS):
        frame[f"mix_prob_{formation}"] = probs[:, idx]
    frame["mix_entropy"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    frame["mix_top1_prob"] = sorted_probs[:, 0]
    frame["mix_top2_prob"] = sorted_probs[:, 1]
    frame["mix_prob_margin"] = frame["mix_top1_prob"] - frame["mix_top2_prob"]
    frame["mix_confidence"] = 1.0 / (1.0 + np.exp((frame["mix_entropy"] - 1.15) / 0.12))
    return frame


train_df = add_regime_probabilities(train_df)
test_df = add_regime_probabilities(test_df)

X_base = train_df[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
X_base_test = test_df[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
y_delta = train_df["target_delta"].astype(float)

global_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=520,
    learning_rate=0.036,
    max_depth=5,
    min_child_weight=90,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.20,
    reg_lambda=8.0,
    gamma=0.10,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)
global_model.fit(X_base, y_delta, verbose=False)
global_delta = global_model.predict(X_base_test)

def fit_weighted_ridge(X_train, y_train, sample_weight, alpha):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    ridge = Ridge(alpha=alpha)
    X_imp = imputer.fit_transform(X_train)
    X_scaled = scaler.fit_transform(X_imp)
    ridge.fit(X_scaled, y_train, sample_weight=sample_weight)
    return {"imputer": imputer, "scaler": scaler, "ridge": ridge}


def predict_weighted_ridge(model, X):
    return model["ridge"].predict(model["scaler"].transform(model["imputer"].transform(X)))


X_regime = train_df[REGIME_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
X_regime_test = test_df[REGIME_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
train_probs = train_df[[f"mix_prob_{formation}" for formation in FORMATION_COLS]].to_numpy(float)
test_probs = test_df[[f"mix_prob_{formation}" for formation in FORMATION_COLS]].to_numpy(float)

regime_test_preds = []
for idx, formation in enumerate(FORMATION_COLS):
    weights = np.clip(train_probs[:, idx], 1e-4, None)
    weights = weights / np.mean(weights)
    model = fit_weighted_ridge(X_regime, y_delta, weights, alpha=80.0)
    regime_test_preds.append(predict_weighted_ridge(model, X_regime_test))
regime_delta = np.sum(test_probs * np.column_stack(regime_test_preds), axis=1)

confidence = test_df["mix_confidence"].fillna(test_df["mix_confidence"].median()).clip(0.0, 1.0)
margin = test_df["mix_prob_margin"].fillna(test_df["mix_prob_margin"].median()).clip(0.0, 1.0)
entropy = test_df["mix_entropy"].fillna(test_df["mix_entropy"].median())
entropy_score = 1.0 / (1.0 + np.exp((entropy - 1.15) / 0.12))
blend_weight = np.clip(0.50 * confidence + 0.35 * margin + 0.15 * entropy_score, 0.0, 1.0)
pred_delta = (1.0 - blend_weight.to_numpy(float)) * global_delta + blend_weight.to_numpy(float) * regime_delta
test_df["pred_tvt"] = test_df["last_known_tvt_feature"].to_numpy(float) + pred_delta

print("train_wells:", len(train_wells))
print("test_wells:", len(test_wells))
print("train_rows:", len(train_df))
print("test_rows:", len(test_df))
print("base_features:", len(BASE_FEATURE_COLS))
print("regime_features:", len(REGIME_FEATURE_COLS))
print("mean_blend_weight:", float(np.mean(blend_weight)))

## 4. Submission

In [ ]:
"Generate predictions in sample-submission order and write submission.csv."

prediction_map = {
    f"{row.wellname}_{int(row.row_idx_original)}": float(row.pred_tvt)
    for row in test_df[["wellname", "row_idx_original", "pred_tvt"]].itertuples(index=False)
}

submission = sample[["id"]].copy()
submission["tvt"] = submission["id"].map(prediction_map)

assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["tvt"].notna().all()
assert np.isfinite(submission["tvt"].to_numpy()).all()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print("submission_rows:", len(submission))
print("submission_path:", OUTPUT_PATH)
print("prediction_min:", float(submission["tvt"].min()))
print("prediction_max:", float(submission["tvt"].max()))
submission.head()